# Memory Management in Agentic AI

## The Setup: 3 Agents × 4 Tools Each

```
👤 User Query
     ↓
🧠 Orchestrator / Router
     ├──► Agent 1: Research Agent  → [Web Search, Wikipedia API, ArXiv Search, URL Scraper]
     ├──► Agent 2: Medical Agent   → [Patient DB Query, Drug Interaction Check, Lab Results Parser, Risk Calculator]
     └──► Agent 3: Report Agent    → [PDF Generator, Chart Creator, Email Sender, Summary Writer]
```

---

## Memory Types — NOT LSTM

Agentic AI does **not** use LSTM. It uses **4 distinct memory stores**:

```
┌─────────────────────────────────────────────────────────────────┐
│                    MEMORY ARCHITECTURE                          │
│                                                                 │
│  ┌─────────────┐  ┌─────────────┐  ┌──────────┐  ┌─────────┐  │
│  │ In-Context  │  │  Short-Term │  │Long-Term │  │Episodic │  │
│  │  (Working)  │  │  (Session)  │  │(External)│  │(History)│  │
│  │             │  │             │  │          │  │         │  │
│  │ LLM prompt  │  │ Redis/RAM   │  │ VectorDB │  │ SQL DB  │  │
│  │ window      │  │ K-V Store   │  │ ChromaDB │  │ MongoDB │  │
│  │ ~128k tokens│  │ minutes-hrs │  │ forever  │  │ forever │  │
│  └─────────────┘  └─────────────┘  └──────────┘  └─────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

| Memory Type | Technology | Lifespan | What It Stores |
|---|---|---|---|
| **In-Context** | LLM prompt window | Current call only | Active reasoning, tool results |
| **Short-Term (STM)** | Redis / in-memory dict | Session (minutes–hours) | Conversation history, agent state |
| **Long-Term (LTM)** | Vector DB (ChromaDB/Pinecone) | Permanent | Knowledge, past patient records |
| **Episodic** | SQL / MongoDB | Permanent | Full past runs, audit trail |

---

## Why NOT LSTM?

| Approach | Problem | Why Agentic AI Doesn't Use It |
|---|---|---|
| **LSTM** | Sequential, fixed hidden state | Can't access external databases, scales poorly |
| **RNN** | Vanishing gradient, slow | Can't do parallel tool calls |
| **Transformers alone** | Limited context window | 128k tokens fills up fast in multi-agent runs |
| **Modern Agentic** | External memory stores | Unlimited, queryable, shareable across agents ✅ |

```
LSTM memory:   [h₀] → [h₁] → [h₂] → [h₃]   fixed size, sequential
               ← information degrades over distance →

Agentic memory: Redis  (instant read/write)
              + VectorDB (semantic search over millions of docs)
              + SQL     (structured queries)
              = Unlimited, precise, shareable, queryable
```

## Full Context Flow — Step by Step

```
User: "Research diabetes + create patient report for John"
         ↓
         STM WRITE: session_id → {query, timestamp}
         ↓
    🧠 Orchestrator
    Reads: STM + LTM
    Plans: [1. ResearchAgent, 2. MedicalAgent, 3. ReportAgent]
         ↓
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 AGENT 1: Research Agent
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Receives Context Packet: {user_goal, session_id, prior_results=[]}
   ↓
   Tool 1: Web Search      → "latest diabetes treatment 2026"
   Tool 2: ArXiv Search    → "diabetes LLM clinical trials"
   Tool 3: Wikipedia API   → "Metformin mechanism of action"
   Tool 4: URL Scraper     → scrape top 2 articles
   ↓
   STM WRITE: session[research_results] = {articles, summary}
         ↓
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 AGENT 2: Medical Agent
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Receives Context Packet: {user_goal, session_id, A1_output=research_summary}
   ↓
   Tool 5: Patient DB Query     → LTM VectorDB: "John diabetes history"
   Tool 6: Drug Interaction     → check Metformin + Amlodipine
   Tool 7: Lab Results Parser   → parse John's HbA1c trend
   Tool 8: Risk Calculator      → compute 10-yr CVD risk
   ↓
   STM WRITE: session[medical_results] = {diagnosis, risk, recommendation}
         ↓
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 AGENT 3: Report Agent
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Receives Full Context Packet: {user_goal, A1_output, A2_output}
   ↓
   Tool 9:  Summary Writer  → combine A1 + A2
   Tool 10: Chart Creator   → BP trend chart
   Tool 11: PDF Generator   → compile report
   Tool 12: Email Sender    → send to doctor
   ↓
   Episodic DB WRITE: {session_id, all_steps, final_output, timestamp}
   STM DELETE: session expired
         ↓
✅ Final Report → User
```

---

## The Context Packet — What Gets Passed Between Agents

Every agent receives and updates a **Context Packet**:

```python
context_packet = {
    "session_id": "sess_abc123",          # ties everything together in Redis
    "user_goal": "Research diabetes + report for John",
    "user_id": "doctor_42",

    # Memory references
    "stm_key": "redis://sess_abc123",      # pointer to short-term memory
    "ltm_collection": "patient_records",  # pointer to vector DB collection

    # Results accumulate here as agents run
    "completed_steps": [
        {
            "agent": "ResearchAgent",
            "tools_used": ["WebSearch", "ArXiv"],
            "output_summary": "Diabetes treatments: Metformin first-line...",
            "output_key": "stm://sess_abc123/research"
        }
    ],

    # Current agent fills this
    "current_agent": "MedicalAgent",
    "current_task": "Analyze John's vitals against research findings"
}
```

## Concrete Example — Full Memory Trace

**User says**: `"Check if John's new BP medication interacts with his diabetes drugs and write a report"`

```
TICK 0 — User Input
──────────────────────────────────────────────────────────
Redis WRITE:  sess_001 = {
    query: "Check BP medication interaction for John",
    user:  "Dr. Smith",  timestamp: "2026-06-11T10:00:00"
}

TICK 1 — Orchestrator reads context
──────────────────────────────────────────────────────────
Redis READ: sess_001
LLM Prompt (In-Context window):
  [SYSTEM]    You are an orchestrator...
  [USER GOAL] Check BP medication interaction for John
  [HISTORY]   (empty - first run)
  → Decision: MedicalAgent → ResearchAgent → ReportAgent

TICK 2 — Medical Agent runs
──────────────────────────────────────────────────────────
VectorDB READ: similarity_search("John patient records")
  → John, 45M, Amlodipine 5mg since Jan 2026, Metformin 500mg since 2024

Tool: Drug Interaction Check("Amlodipine", "new_BP_drug")
  → "Moderate interaction — monitor potassium levels"

Redis WRITE: sess_001:medical = {
    patient: "John",
    current_drugs: ["Amlodipine", "Metformin"],
    interaction: "Moderate — monitor potassium",
    recommendation: "Reduce Amlodipine to 2.5mg"
}

TICK 3 — Research Agent reads Medical Agent output
──────────────────────────────────────────────────────────
Redis READ: sess_001:medical   ← reads what Medical Agent wrote

Tool: Web Search("Amlodipine interaction guidelines 2026")
Tool: ArXiv Search("calcium channel blocker diabetes combination therapy")

Redis WRITE: sess_001:research = {
    guideline: "ACC 2025: Monitor eGFR when combining CCB + RAAS in diabetics",
    evidence: "Strong evidence for dose reduction in T2DM patients"
}

TICK 4 — Report Agent compiles everything
──────────────────────────────────────────────────────────
Redis READ ALL: sess_001:*  → gets medical + research results

LLM In-Context receives SUMMARY (not full raw data):
  "Medical: Moderate interaction, reduce Amlodipine..."
  "Research: ACC 2025 guideline recommends..."

Tool: Summary Writer → merges both findings
Tool: PDF Generator  → creates final report

MongoDB WRITE: {              ← Episodic memory (permanent)
    session_id: "sess_001",
    patient: "John",
    query: "Check BP medication interaction...",
    report_path: "/reports/john_2026-06-11.pdf",
    all_steps: [...],
    duration_ms: 8420
}

Redis DELETE: sess_001:*       ← STM cleared after session ends
✅ Report delivered to Dr. Smith
```

## Step 1 — Install Dependencies

In [ ]:
!pip install redis langchain langchain-openai chromadb pymongo sentence-transformers

## Step 2 — Short-Term Memory (STM) with Redis

STM stores session-scoped data — alive only during the current conversation.  
Every agent reads and writes to the same Redis session key.

```
Redis Key Pattern:  {session_id}:{agent_name}
TTL:                3600 seconds (1 hour) — auto-deleted after session
```

In [ ]:
import json
import uuid
from datetime import datetime

# ── Simulated STM using dict (replace with redis.Redis() in production) ──
class ShortTermMemory:
    """
    In production: replace self._store with redis.Redis(host='localhost', port=6379)
    and use r.set(key, json.dumps(value), ex=3600) / r.get(key)
    """
    def __init__(self):
        self._store = {}  # simulates Redis in-memory

    def write(self, session_id: str, key: str, value: dict, ttl_seconds: int = 3600):
        full_key = f"{session_id}:{key}"
        self._store[full_key] = {"data": value, "expires_at": ttl_seconds}
        print(f"  STM WRITE → {full_key}")

    def read(self, session_id: str, key: str) -> dict:
        full_key = f"{session_id}:{key}"
        entry = self._store.get(full_key)
        if entry:
            print(f"  STM READ  ← {full_key}")
            return entry["data"]
        return {}

    def read_all(self, session_id: str) -> dict:
        result = {}
        for key, val in self._store.items():
            if key.startswith(session_id):
                short_key = key.replace(f"{session_id}:", "")
                result[short_key] = val["data"]
        return result

    def delete_session(self, session_id: str):
        keys_to_del = [k for k in self._store if k.startswith(session_id)]
        for k in keys_to_del:
            del self._store[k]
        print(f"  STM DELETE → session '{session_id}' cleared ({len(keys_to_del)} keys)")


stm = ShortTermMemory()

# ── Test STM ────────────────────────────────────────────────
session_id = "sess_" + str(uuid.uuid4())[:8]
print(f"Session ID: {session_id}\n")

stm.write(session_id, "context", {
    "user_goal": "Check BP medication interaction for John",
    "user": "Dr. Smith",
    "timestamp": datetime.now().isoformat()
})

data = stm.read(session_id, "context")
print(f"\nRead back: {data}")

## Step 3 — Long-Term Memory (LTM) with VectorDB (ChromaDB)

LTM stores permanent knowledge — patient records, medical guidelines, past findings.  
Uses **semantic similarity search** — find records by meaning, not exact keyword match.

```
VectorDB stores:
  "John patient history 2024"  → embedding vector [0.23, -0.45, 0.87, ...]
  "Metformin side effects"     → embedding vector [0.11,  0.32, -0.55, ...]

Query: "John diabetes records"
  → converts to embedding → finds closest vectors → returns relevant docs
```

In [ ]:
import chromadb

# ── Long-Term Memory using ChromaDB ─────────────────────────
class LongTermMemory:
    def __init__(self, collection_name: str = "patient_records"):
        self.client = chromadb.Client()  # in-memory; use PersistentClient("./ltm_db") for disk
        self.collection = self.client.get_or_create_collection(collection_name)

    def store(self, doc_id: str, text: str, metadata: dict = {}):
        self.collection.add(
            documents=[text],
            metadatas=[metadata],
            ids=[doc_id]
        )
        print(f"  LTM WRITE → '{doc_id}'")

    def search(self, query: str, top_k: int = 3) -> list:
        results = self.collection.query(query_texts=[query], n_results=top_k)
        docs = results["documents"][0]
        print(f"  LTM SEARCH ← '{query}' → {len(docs)} results")
        return docs


ltm = LongTermMemory("patient_records")

# ── Seed patient records into LTM ───────────────────────────
patient_records = [
    ("john_bp_2024",     "John 45M BP history: Jan 2024: 145/92, Jun 2024: 150/95, Dec 2024: 148/93",   {"patient": "John", "type": "bp_history"}),
    ("john_hba1c_2024",  "John HbA1c trend: Jan 2024: 8.2%, Jun 2024: 7.9%, Dec 2024: 8.1%",            {"patient": "John", "type": "lab_results"}),
    ("john_meds_2026",   "John current medications: Amlodipine 5mg (since Jan 2026), Metformin 500mg (since 2024)", {"patient": "John", "type": "medications"}),
    ("drug_interaction", "Amlodipine + Metformin: No major interaction. Monitor potassium. Safe combination.", {"type": "drug_db"}),
    ("diabetes_guide",   "Diabetes first-line treatment 2026: Metformin 500mg BID. Target HbA1c < 7.0%", {"type": "guideline"}),
]

print("Seeding LTM with patient records...\n")
for doc_id, text, meta in patient_records:
    ltm.store(doc_id, text, meta)

# ── Test LTM search ──────────────────────────────────────────
print("\n--- LTM Search Test ---")
results = ltm.search("John diabetes medication history")
for i, r in enumerate(results):
    print(f"  Result {i+1}: {r[:80]}...")

## Step 4 — Episodic Memory (MongoDB)

Episodic memory permanently records the **full history of every agent run** — for audit, debugging, and learning from past sessions.

In [ ]:
from datetime import datetime

# ── Simulated Episodic Memory (replace with pymongo in production) ──
class EpisodicMemory:
    """
    In production:
        from pymongo import MongoClient
        client = MongoClient("mongodb://localhost:27017/")
        db = client["agentic_ai"]
        self.collection = db["episodes"]
    """
    def __init__(self):
        self._store = []  # simulates MongoDB collection

    def save_episode(self, session_id: str, user_query: str, all_steps: list, final_output: str):
        episode = {
            "_id": session_id,
            "user_query": user_query,
            "all_steps": all_steps,
            "final_output": final_output,
            "timestamp": datetime.now().isoformat(),
            "step_count": len(all_steps)
        }
        self._store.append(episode)
        print(f"  EPISODIC WRITE → session '{session_id}' ({len(all_steps)} steps saved)")
        return episode

    def get_past_episodes(self, patient: str = None) -> list:
        if patient:
            return [e for e in self._store if patient.lower() in e["user_query"].lower()]
        return self._store


episodic = EpisodicMemory()
print("✅ Episodic memory ready")

## Step 5 — The 3 Agents with 4 Tools Each

Now we build all 3 agents. Each agent:
1. Reads the context packet from STM
2. Reads patient data from LTM (VectorDB)
3. Runs its 4 tools
4. Writes results back to STM

In [ ]:
import time

# ════════════════════════════════════════════════════════════
# AGENT 1: Research Agent — 4 Tools
# ════════════════════════════════════════════════════════════
class ResearchAgent:
    def __init__(self, stm: ShortTermMemory):
        self.stm = stm
        self.name = "ResearchAgent"

    # Tool 1: Web Search (simulated)
    def tool_web_search(self, query: str) -> dict:
        print(f"    [Tool 1 - Web Search] query='{query}'")
        return {"source": "web", "result": f"Latest guidelines on '{query}': Metformin first-line, GLP-1 second-line (2026 ACC)"}

    # Tool 2: Wikipedia API (simulated)
    def tool_wikipedia(self, topic: str) -> dict:
        print(f"    [Tool 2 - Wikipedia] topic='{topic}'")
        return {"source": "wikipedia", "result": f"'{topic}' — biguanide class drug, reduces hepatic glucose production"}

    # Tool 3: ArXiv Search (simulated)
    def tool_arxiv(self, query: str) -> dict:
        print(f"    [Tool 3 - ArXiv] query='{query}'")
        return {"source": "arxiv", "result": f"Paper: 'LLM-assisted diabetes management 2025' — 94% accuracy in Tx recommendations"}

    # Tool 4: URL Scraper (simulated)
    def tool_url_scraper(self, url: str) -> dict:
        print(f"    [Tool 4 - URL Scraper] url='{url}'")
        return {"source": url, "result": "Scraped: Diabetes treatment protocol v3.2 — updated June 2026"}

    def run(self, session_id: str) -> dict:
        print(f"\n{'━'*55}")
        print(f"  AGENT 1: {self.name} starting...")
        print(f"{'━'*55}")

        # Read context from STM
        ctx = self.stm.read(session_id, "context")

        # Run 4 tools
        r1 = self.tool_web_search("diabetes treatment 2026")
        r2 = self.tool_wikipedia("Metformin")
        r3 = self.tool_arxiv("diabetes LLM clinical")
        r4 = self.tool_url_scraper("https://acc.org/diabetes-2026")

        output = {
            "agent": self.name,
            "tools_used": ["WebSearch", "Wikipedia", "ArXiv", "URLScraper"],
            "summary": "Metformin remains first-line. GLP-1 agonists for HbA1c > 8%. Monitor BP in T2DM.",
            "raw_results": [r1, r2, r3, r4]
        }

        # Write to STM so next agent can read it
        self.stm.write(session_id, self.name, output)
        print(f"  ✅ {self.name} done — results stored in STM")
        return output


# ════════════════════════════════════════════════════════════
# AGENT 2: Medical Agent — 4 Tools
# ════════════════════════════════════════════════════════════
class MedicalAgent:
    def __init__(self, stm: ShortTermMemory, ltm: LongTermMemory):
        self.stm = stm
        self.ltm = ltm
        self.name = "MedicalAgent"

    # Tool 5: Patient DB Query (via LTM VectorDB)
    def tool_patient_db(self, patient: str) -> dict:
        print(f"    [Tool 5 - Patient DB] searching for '{patient}'")
        records = self.ltm.search(f"{patient} medical history medications")
        return {"patient": patient, "records": records}

    # Tool 6: Drug Interaction Check (simulated)
    def tool_drug_interaction(self, drug1: str, drug2: str) -> dict:
        print(f"    [Tool 6 - Drug Interaction] {drug1} + {drug2}")
        return {"drugs": [drug1, drug2], "interaction": "Moderate — monitor potassium levels",
                "recommendation": f"Reduce {drug1} dose to 2.5mg if adding new drug"}

    # Tool 7: Lab Results Parser (simulated)
    def tool_lab_parser(self, patient: str) -> dict:
        print(f"    [Tool 7 - Lab Parser] parsing labs for '{patient}'")
        return {"hba1c_trend": "8.2% → 7.9% → 8.1% (slightly worsening)", "bp_trend": "145/92 → 150/95 (rising)"}

    # Tool 8: Risk Calculator (simulated)
    def tool_risk_calc(self, age: int, smoker: bool, bp: str, diabetes: bool) -> dict:
        print(f"    [Tool 8 - Risk Calculator] age={age}, smoker={smoker}, bp={bp}")
        risk_score = 22 + (5 if smoker else 0) + (age - 40) * 0.5
        return {"10yr_cvd_risk": f"{risk_score:.0f}%", "level": "HIGH" if risk_score > 20 else "MODERATE"}

    def run(self, session_id: str, patient: str) -> dict:
        print(f"\n{'━'*55}")
        print(f"  AGENT 2: {self.name} starting...")
        print(f"{'━'*55}")

        # Read context + Research Agent output from STM
        ctx      = self.stm.read(session_id, "context")
        research = self.stm.read(session_id, "ResearchAgent")  # ← reads Agent 1 output

        # Run 4 tools
        r5 = self.tool_patient_db(patient)
        r6 = self.tool_drug_interaction("Amlodipine", "new_BP_drug")
        r7 = self.tool_lab_parser(patient)
        r8 = self.tool_risk_calc(age=45, smoker=False, bp="155/95", diabetes=True)

        output = {
            "agent": self.name,
            "tools_used": ["PatientDB", "DrugInteraction", "LabParser", "RiskCalc"],
            "patient": patient,
            "diagnosis": "Type 2 Diabetes + Stage 1 Hypertension",
            "drug_interaction": r6["interaction"],
            "recommendation": "Reduce Amlodipine to 2.5mg. Continue Metformin 500mg. Recheck in 2 weeks.",
            "risk": r8,
            "research_applied": research.get("summary", "")
        }

        self.stm.write(session_id, self.name, output)
        print(f"  ✅ {self.name} done — results stored in STM")
        return output


# ════════════════════════════════════════════════════════════
# AGENT 3: Report Agent — 4 Tools
# ════════════════════════════════════════════════════════════
class ReportAgent:
    def __init__(self, stm: ShortTermMemory, episodic: EpisodicMemory):
        self.stm      = stm
        self.episodic = episodic
        self.name     = "ReportAgent"

    # Tool 9: Summary Writer
    def tool_summary_writer(self, research: dict, medical: dict) -> str:
        print(f"    [Tool 9 - Summary Writer] combining agent outputs")
        return (f"Patient {medical['patient']}: {medical['diagnosis']}. "
                f"Drug alert: {medical['drug_interaction']}. "
                f"Action: {medical['recommendation']}. "
                f"Evidence: {research.get('summary', 'N/A')}")

    # Tool 10: Chart Creator (simulated)
    def tool_chart_creator(self, trend_data: str) -> str:
        print(f"    [Tool 10 - Chart Creator] generating BP trend chart")
        return "chart_bp_trend_john_2026.png (generated)"

    # Tool 11: PDF Generator (simulated)
    def tool_pdf_generator(self, content: str, patient: str) -> str:
        print(f"    [Tool 11 - PDF Generator] creating report PDF")
        return f"/reports/{patient.lower()}_medical_report_2026.pdf"

    # Tool 12: Email Sender (simulated)
    def tool_email_sender(self, to: str, report_path: str) -> str:
        print(f"    [Tool 12 - Email Sender] sending to {to}")
        return f"Email sent to {to} with attachment {report_path}"

    def run(self, session_id: str) -> dict:
        print(f"\n{'━'*55}")
        print(f"  AGENT 3: {self.name} starting...")
        print(f"{'━'*55}")

        # Read ALL agent outputs from STM
        research = self.stm.read(session_id, "ResearchAgent")
        medical  = self.stm.read(session_id, "MedicalAgent")

        # In-Context: LLM receives a SUMMARY (not full raw data — saves token space)
        in_context_summary = {
            "research_summary": research.get("summary"),
            "diagnosis":        medical.get("diagnosis"),
            "recommendation":   medical.get("recommendation"),
            "drug_alert":       medical.get("drug_interaction"),
            "risk":             medical.get("risk")
        }
        print(f"    In-Context window receives: {list(in_context_summary.keys())}")

        # Run 4 tools
        r9  = self.tool_summary_writer(research, medical)
        r10 = self.tool_chart_creator("BP: 145→150→155")
        r11 = self.tool_pdf_generator(r9, medical["patient"])
        r12 = self.tool_email_sender("dr.smith@hospital.com", r11)

        output = {
            "agent":        self.name,
            "tools_used":   ["SummaryWriter", "ChartCreator", "PDFGenerator", "EmailSender"],
            "final_summary": r9,
            "report_path":   r11,
            "email_status":  r12
        }

        self.stm.write(session_id, self.name, output)

        # ── Archive to Episodic memory ────────────────────────
        all_steps = [
            self.stm.read(session_id, "ResearchAgent"),
            self.stm.read(session_id, "MedicalAgent"),
            output
        ]
        ctx = self.stm.read(session_id, "context")
        self.episodic.save_episode(session_id, ctx.get("user_goal", ""), all_steps, r9)

        print(f"  ✅ {self.name} done — report generated and archived")
        return output


print("✅ All 3 agent classes defined")

## Step 6 — Orchestrator + Full Run

The orchestrator coordinates all 3 agents, passing context between them via STM.

In [ ]:
import uuid, time

def orchestrator(user_query: str, patient_name: str):
    """
    Orchestrator coordinates all agents:
    1. Creates session in STM
    2. Runs agents in sequence (ResearchAgent → MedicalAgent → ReportAgent)
    3. Passes context via STM between agents
    4. Archives final run to Episodic memory
    5. Clears STM after session
    """
    session_id = "sess_" + str(uuid.uuid4())[:8]
    start_time = time.time()

    print("=" * 60)
    print(f"🧠 ORCHESTRATOR STARTED")
    print(f"   Session ID  : {session_id}")
    print(f"   User Query  : {user_query}")
    print(f"   Patient     : {patient_name}")
    print("=" * 60)

    # ── Step 0: Initialize session in STM ───────────────────
    stm.write(session_id, "context", {
        "user_goal": user_query,
        "patient": patient_name,
        "timestamp": datetime.now().isoformat()
    })

    # ── Step 1: Research Agent ───────────────────────────────
    agent1 = ResearchAgent(stm)
    agent1.run(session_id)

    # ── Step 2: Medical Agent (reads Agent 1 output via STM) ─
    agent2 = MedicalAgent(stm, ltm)
    agent2.run(session_id, patient_name)

    # ── Step 3: Report Agent (reads all outputs via STM) ─────
    agent3 = ReportAgent(stm, episodic)
    final_output = agent3.run(session_id)

    # ── Step 4: Clear STM (session over) ─────────────────────
    stm.delete_session(session_id)

    duration = round((time.time() - start_time) * 1000)

    print(f"\n{'='*60}")
    print(f"✅ ORCHESTRATOR COMPLETE — {duration}ms")
    print(f"   Final Report : {final_output['report_path']}")
    print(f"   Summary      : {final_output['final_summary'][:80]}...")
    print(f"   Episodic DB  : session '{session_id}' archived")
    print("=" * 60)
    return final_output


# ── RUN IT ────────────────────────────────────────────────────
result = orchestrator(
    user_query   = "Check John's BP medication interaction with his diabetes drugs and write a report",
    patient_name = "John"
)

## Step 7 — Inspect Episodic Memory (Past Runs)

In [ ]:
# View all past episodes (persistent across sessions)
past_runs = episodic.get_past_episodes()
print(f"Total episodes stored: {len(past_runs)}\n")

for ep in past_runs:
    print(f"Session  : {ep['_id']}")
    print(f"Query    : {ep['user_query']}")
    print(f"Steps    : {ep['step_count']} agent steps")
    print(f"Output   : {ep['final_output'][:100]}...")
    print(f"Time     : {ep['timestamp']}")
    print("-" * 50)

# Search for specific patient's history
print("\nPast episodes for 'John':")
john_episodes = episodic.get_past_episodes(patient="John")
print(f"Found {len(john_episodes)} episode(s)")

# Summary — Complete Memory Architecture

## All 4 Memory Types at a Glance

| Memory | Technology | Scope | Read By | Written By |
|---|---|---|---|---|
| **In-Context** | LLM prompt window | Single LLM call | LLM internally | Agent builds prompt |
| **STM** | Redis (K-V store) | One session | All 3 agents | Each agent after its run |
| **LTM** | ChromaDB / Pinecone | Permanent | Medical Agent (mostly) | Pre-seeded + ongoing |
| **Episodic** | MongoDB / SQL | Permanent | Orchestrator, audit | Report Agent (end of run) |

## Data Flow Between Agents

```
User Query
    ↓
Orchestrator ──writes──► STM: {session_id}:context
    ↓
Agent 1 (Research)
    ├── reads  STM: context
    ├── runs   4 tools
    └── writes STM: {session_id}:ResearchAgent
    ↓
Agent 2 (Medical)
    ├── reads  STM: context + ResearchAgent output
    ├── reads  LTM: patient history (VectorDB)
    ├── runs   4 tools
    └── writes STM: {session_id}:MedicalAgent
    ↓
Agent 3 (Report)
    ├── reads  STM: context + ResearchAgent + MedicalAgent
    ├── LLM In-Context: receives SUMMARY (not full raw data)
    ├── runs   4 tools
    ├── writes Episodic: full session archived to MongoDB
    └── STM cleared ← session over
    ↓
✅ Final answer to user
```

## Key Insight

> **The LLM's context window (In-Context memory) never holds everything.**  
> It only holds a **summary** of what's in STM/LTM.  
> The actual data lives in Redis + VectorDB + MongoDB.  
> This is how agents handle conversations with unlimited history  
> without ever exceeding the 128k token limit.